In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import pandas as pd
import os
import matplotlib.pyplot as plt
import cortex
import seaborn as sns
from os.path import join
from collections import defaultdict
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import dvu
from neuro.flatmaps_helper import load_flatmaps
from neuro.features.questions.gpt4 import QS_35_STABLE
import sys
import warnings
sys.path.append('../notebooks')
from tqdm import tqdm
from neuro import config
from neuro import analyze_helper
import neuro.viz
from neuro.features.qa_questions import get_questions, get_merged_questions_v3_boostexamples
# flatmaps_per_question = __import__('06_flatmaps_per_question')
import viz
import gct
from neuro.flatmaps_helper import load_flatmaps
from statsmodels.stats.multitest import multipletests

Note, this notebook requires first running `03_export_qa_flatmaps.ipynb` into `df_qa_dict.pkl` files for each subject.

In [ ]:
qa_questions_list = QS_35_STABLE
subject = 'S02'

In [ ]:
# load qa weights
corrs_df_dict = {}
frac_voxels_to_keep_list = [0.01, 0.05, 0.1, 0.25, 0.5, 1]

# corrs used for masking
corrs_test = joblib.load(join(config.PROCESSED_DIR, subject.replace(
    'UT', ''), 'corrs_test_35.pkl')).values[0]
corrs_test_individual_dict = joblib.load(join(config.PROCESSED_DIR, subject.replace(
    'UT', ''), 'corrs_test_individual_gpt4_qs_35.pkl'))

In [ ]:
def mask_voxels(df, frac_voxels_to_keep, corrs_mask_per_question=True):
    df = df.copy()
    if frac_voxels_to_keep < 1:
        # mask based on corrs
        if corrs_mask_per_question:
            for i in range(df.shape[0]):
                q = df.index[i]
            # q = questions_names_df['qa'].values[i]
                mask = (corrs_test_individual_dict[q] > np.percentile(
                    corrs_test_individual_dict[q], 100 * (1 - frac_voxels_to_keep))).astype(bool)
                for col in range(len(df.columns)):
                    df.iloc[i, col] = df.iloc[i, col][mask]
        else:
            mask = (corrs_test > np.percentile(
                corrs_test, 100 * (1 - frac_voxels_to_keep))).astype(bool)

            for i in range(df.shape[0]):
                for col in range(len(df.columns)):
                    df.iloc[i, col] = df.iloc[i, col][mask]
    return df

def compute_corrs_with_first_col(df):
    correlations_to_first = defaultdict(list)
    for col_idx in range(1, df.shape[1]):
        corrs = []
        for i in range(df.shape[0]):
            corrs.append(np.corrcoef(
                df.iloc[i, 0],
                df.iloc[i, col_idx]
            )[0, 1])
        correlations_to_first[col_idx] = corrs
    d = pd.DataFrame(correlations_to_first, index=QS_35_STABLE)
    d.columns = df.columns[1:]
    return d

In [ ]:
settings = [
    'individual_gpt4',
    'full_35_gpt4_pc',
    'individual_gpt4_pc_new',
    'individual_gpt4_wordrate', 
]
frac_voxels_to_keep = 0.5

flatmap_lists_by_setting = defaultdict(list)
for setting in tqdm(settings): 
    # setting = 'individual_gpt4_pc_new'
    flatmaps_qa_dict = joblib.load(
        join(config.PROCESSED_DIR, subject.replace('UT', ''), setting + '.pkl'))
    for q in QS_35_STABLE:
        flatmap = flatmaps_qa_dict[q]
        flatmap_lists_by_setting[setting].append(flatmaps_qa_dict[q])


df_flatmap_by_setting_main = pd.DataFrame(flatmap_lists_by_setting, index=QS_35_STABLE)
df_flatmap_by_setting_25 = mask_voxels(
    df_flatmap_by_setting_main, frac_voxels_to_keep=0.25, corrs_mask_per_question=True)
df_flatmap_by_setting_1 = mask_voxels(
    df_flatmap_by_setting_main, frac_voxels_to_keep=0.1, corrs_mask_per_question=True)
df_flatmap_by_setting_01 = mask_voxels(
    df_flatmap_by_setting_main, frac_voxels_to_keep=0.01, corrs_mask_per_question=True)


# compute the correlation between flatmaps of every column to the first column
d = compute_corrs_with_first_col(df_flatmap_by_setting_main)
d_25 = compute_corrs_with_first_col(df_flatmap_by_setting_25)
d_1 = compute_corrs_with_first_col(df_flatmap_by_setting_1)
d_01 = compute_corrs_with_first_col(df_flatmap_by_setting_01)


d_full = pd.concat(
    (
    pd.DataFrame(d.mean(axis=0)).T,
    pd.DataFrame(d_25.mean(axis=0)).T,
    pd.DataFrame(d_1.mean(axis=0)).T,
    pd.DataFrame(d_01.mean(axis=0)).T,
    d_1.sort_values(by=d_1.columns[-1], ascending=False)
    ),
    ignore_index=False
).round(3)
d_full.index = [
    'AVG (all voxels)',
    'AVG (25% top-predicted)',
    'AVG (10% top-predicted)',
    'AVG (1% top-predicted)'
] + list(d.index)
d_full = d_full.rename(columns={
    'full_35_gpt4_pc': '+Response PCs, +Joint fitting',
    'individual_gpt4_pc_new': '+Response PCs',
    'individual_gpt4_wordrate': '+Word rate',
})
# reverse col order
d_full = d_full[d_full.columns[::-1]]
# d_full.to_latex()
d_full

In [ ]:
d_full.index = d_full.index.map(neuro.analyze_helper.abbrev_question)
print(d_full.style.format(precision=3).to_latex(hrules=True).replace('%', '\%'))

In [ ]:
flatmaps_location = df_flatmap_by_setting_main.loc['Does the sentence mention a specific location?']

for k in flatmaps_location.index:
    print(k)
    neuro.viz.quickshow(
        flatmaps_location[k].flatten(),
        fname_save=join('compare_variations', f'{k.replace(" ", "_")}_{subject}.png'),
        kwargs={'with_rois': False},
        with_colorbar=False,)